# GEMAMX GROMACS - Ejecucion MD en Colab

Libreta base para correr MD en Colab y guardar `gromacs/results/gromacs_metrics.json` en Google Drive.
Esta version deja preparado el contrato de salida aunque la instalacion final de GROMACS/MM-GBSA se ajuste segun el entorno.


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
%cd /content/drive/MyDrive/GEMAMX


In [ ]:
import json
from datetime import datetime
from pathlib import Path

manifest_path = Path('gromacs/results/gromacs_batch_manifest.json')
results_path = Path('gromacs/results/gromacs_metrics.json')
manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
pairs = manifest.get('pairs', [])
print(f'pares seleccionados: {len(pairs)}')


## Instalacion sugerida

Adapta esta celda al runtime real. En Colab Pro o Vertex AI suele convenir una imagen Linux con GROMACS ya disponible.

```bash
# Ejemplo referencial, no universal
# !apt-get update
# !apt-get install -y gromacs
```


In [ ]:
def run_pair_stub(pair):
    tag = pair['tag']
    run_dir = Path('/content/drive/MyDrive/GEMAMX') / pair['recommended_colab_run_dir']
    run_dir.mkdir(parents=True, exist_ok=True)
    return {
        'tag': tag,
        'variant_id': pair['variant_id'],
        'drug_name': pair['drug_name'],
        'vina_score_kcalmol': pair['vina_score_kcalmol'],
        'status': 'pending_manual_gromacs',
        'run_dir': str(run_dir),
        'trajectory_path': None,
        'plot_paths': {},
        'notes': 'Placeholder generado desde Colab. Sustituir con metricas reales de GROMACS.'
    }

results = [run_pair_stub(pair) for pair in pairs]
payload = {
    'generated_at': datetime.now().isoformat(),
    'pipeline_version': 'GEMAMX-GROMACS-v2',
    'parameters': {
        'force_field': 'charmm36m',
        'water_model': 'tip3p',
        'md_time_ns': 50,
        'temperature_K': 300,
        'top_n_per_variant': manifest['selection_policy']['top_n_per_variant']
    },
    'summary': {
        'pairs_total': len(results),
        'pairs_completed': sum(1 for row in results if row['status'] == 'completed'),
        'pairs_failed': sum(1 for row in results if row['status'] in ('failed', 'error'))
    },
    'pairs': results
}
results_path.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding='utf-8')
print(results_path)


## Regreso a local

Ya en Windows, corre:

```bash
python gromacs/ingest_gromacs_results.py
```

Despues recarga `http://localhost:8000/` y abre la pagina `Dinamica Molecular`.
